In [1]:
!pip install -q langgraph transformers torch


In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from transformers import pipeline


class GraphState(TypedDict):
    user_question: str
    clarified_question: str
    final_answer: str



hf_model = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base"
)


def question_analyzer(state: GraphState) -> GraphState:
    prompt = (
        "Rewrite and clarify the following user question:\n\n"
        f"{state['user_question']}"
    )

    clarified = hf_model(prompt, max_length=128)[0]["generated_text"]

    return {
        "clarified_question": clarified
    }


def answer_generator(state: GraphState) -> GraphState:
    prompt = (
        "Answer the following question clearly and concisely:\n\n"
        f"{state['clarified_question']}"
    )

    answer = hf_model(prompt, max_length=256)[0]["generated_text"]

    return {
        "final_answer": answer
    }



graph = StateGraph(GraphState)

graph.add_node("question_analyzer", question_analyzer)
graph.add_node("answer_generator", answer_generator)

graph.set_entry_point("question_analyzer")
graph.add_edge("question_analyzer", "answer_generator")
graph.add_edge("answer_generator", END)

app = graph.compile()



initial_state = {
    "user_question": "What is backpropagation in neural networks?"
}

result = app.invoke(initial_state)

print("Clarified Question:\n", result["clarified_question"])
print("\nFinal Answer:\n", result["final_answer"])


Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clarified Question:
 Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's backpropagation. Backpropagation is the propagation of a neural network's back

Final Answer:
 Backpropagati